In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split

In [ ]:
features = pd.read_parquet('data/diabetes_features.parquet')
target = pd.read_parquet('data/diabetes_target.parquet')

# Creating a first encounter mask
first_encounter_mask = features['encounter_id'] == features.loc[:, ['patient_nbr', 'encounter_id']].groupby('patient_nbr').transform(np.min)['encounter_id']

# Creating features and a target for only the first encounters
first_encounter_features = features[first_encounter_mask]
first_encounter_target = target[first_encounter_mask]

# Splitting train and test
X_train, X_test, y_train, y_test = train_test_split(first_encounter_features, first_encounter_target, test_size=0.2, random_state=42)

In [ ]:
# Creating a dataframe to inspect the split
split_inspection_df = features.loc[:, ['patient_nbr', 'encounter_id']].copy()

# Adding the target to the inspection df
split_inspection_df['target'] = target
split_inspection_df['first_encounter_mask'] = first_encounter_mask
;
# Creating a second dataframe for post split
split_inspection_2 = first_encounter_features.loc[:, ['patient_nbr', 'encounter_id']].copy()
split_inspection_2['target'] = first_encounter_target

# Checking that all targets match
(split_inspection_df[split_inspection_df.first_encounter_mask == True]['target'] == split_inspection_2['target']).astype(int).mean()

# Columns and Data Types

In [ ]:
print(features.info())

In [ ]:
for c in features.columns.values:
    print(c)

In [ ]:
features.patient_nbr.value_counts()

In [ ]:
# Exploring a single patient
features[features.patient_nbr == 88785891]

# Missing Data

#### Key Findings

- Weight is missing for an entire patient, except for 36 patients captured below
- Missing values are encoded in some columns using "?" and None

In [ ]:
# Diagnosing if weight is always missing for a patient
weight_diagnostic = features.loc[:, ['patient_nbr', 'weight']]

weight_diagnostic['missing_weight'] = weight_diagnostic['weight'].apply(lambda x: 1 if x == "?" else 0)

# Group and aggregate
grouped = weight_diagnostic.groupby('patient_nbr').agg({
    'missing_weight': 'sum',  # count of missing weights
    'weight': 'count'          # total records for that patient
})

# Patients where missing_weight == total records are always missing
grouped['always_missing'] = grouped['missing_weight'] == grouped['weight']

grouped[(grouped["missing_weight"] != grouped['weight']) & (grouped["missing_weight"] > 0)]

In [ ]:
# Inspecting one patient that has mixed missing data
X_train[X_train.patient_nbr == 4790016]

### Diving into the use of Identifiers ("?", None) for missing data

In [ ]:
qmark_missing = (features == "?").sum()
print(qmark_missing[qmark_missing > 0])

In [ ]:
none_missing = (features == "None").sum()
print(none_missing[none_missing > 0])

#### Correlation of Missing Values

In [ ]:
((X_train == "?") | (X_train == "None")).astype(int).corr().round(4)

#### Exploring Patterns in Missing Data

Missing data in the following columns depends on admission type and admission source:

- max_glu_serum
- A1Cresult
- weight
- payer_code
- medical_specialty

This pattern is consistent with MAR with dependency on the admission source and type. Options for resolving this are:
- Stratify and fill --> impractical due to the missiness rate (80+%) and some groups not having values
- Flag missingess of the data --> challenge will be the flags will be highly correlated with admission type and source
- Drop columns --> straightforward, but there is information in the fact it was collected
- Filter --> impractical for features with a high missingness rate

My preferred imputation strategy per column is as follows:
- max_glu_serum, A1Cresult, weight, payer_code, medical_specialty --> flag but sanity check correlation with admission cols + conduct feature importance (admission type and source may fully account for this phenomenon in the model)
- race --> impute
- diag_1, diag_2, diag_3 --> drop

In [ ]:
# Admission Type Crosstab Across Key Variables
for f in ['max_glu_serum', 'A1Cresult', 'weight', 'payer_code', 'medical_specialty']:
    ct = pd.crosstab(features['admission_type_id'], features[f])
    print(pd.DataFrame(ct.apply(lambda x: x / sum(x), axis=1)))


In [ ]:
# Admission Source Crosstab Across Key Variables
for f in ['max_glu_serum', 'A1Cresult', 'weight', 'payer_code', 'medical_specialty']:
    ct = pd.crosstab(features['admission_source_id'], features[f])
    print(ct.apply(lambda x: x / sum(x), axis=1))

In [ ]:
# inspecting the specific crosstab of admission source and medical specialty
pd.crosstab(features['admission_source_id'], features['medical_specialty']).apply(lambda x: x / sum(x), axis=1)

## Exploring Categorical Values for Encoding Strategies

Columns for One-Hot Encoding:
- Race
- Gender
- change
- diabetesMed --> binary flag if a patient is taking any of the diabetes meds listed in the dataframe

Columns for Ordinal Encoding:
- Age
- metformin
- insulin

The following columns are highly sparse and will not be included in the training dataset:
- repaglinide
- nateglinide 
- chlorprpamide
- glimepiride
- acetohexamide
- glipizide
- glyburide
- tolbutamide
- pioglitazone
- rosiglitazone
- acarbose 
- miglitol 
- troglitazone 
- tolazamide 
- examide
- citoglipton 
- glyburide.metformin 
- glipizide.metformin 
- glimepiride.pioglitazone
- metformin.rosiglitazone 
- metformin.pioglitazone


In [ ]:
features.select_dtypes(include=['object', 'category']).head(n=20)

In [ ]:
for c in features.select_dtypes(include=['object', 'category']).columns.to_list():
    print(f"Value counts for {c}:")
    print(features[c].value_counts())
    print("\n")

# Outliers

Observations:
- num_lab_procedures has a concentration of observations at a few key values creating "spikiness". Most of this is occuring in admission type ID 1
- number_diagnoses appears to be truncated with a maximum of 9

No additional columns suggest an issue with outliers that can't be solved with scaling. Including admission_type_id should accoutn for the spikiness of lab procedures.

In [ ]:
# Grabbing and melting a dataframe of numeric columns
numeric_cols = X_train.select_dtypes("number")
long_numeric = numeric_cols.melt()

# Plotting a grid of histograms for the numeric columns
grid = sns.FacetGrid(long_numeric, col='variable', col_wrap=4, sharex=False, sharey=False)
grid.map_dataframe(sns.histplot, x='value')
plt.title("Grid of Histograms by Variable")
plt.show()

In [ ]:
# Inspecting the total number of observations by count of lab procedures
numeric_cols['num_lab_procedures'].value_counts().reset_index().sort_values("num_lab_procedures")

In [ ]:
sns.histplot(X_train,
             x="num_lab_procedures",
             hue="admission_type_id",
             multiple='stack',
             palette='Set2')
plt.title("Histogram of Number of Labs by Admission Type")
plt.show()

In [ ]:
num_labs_grid = sns.FacetGrid(X_train, col_wrap=4, col='admission_type_id')
num_labs_grid.map_dataframe(sns.histplot, x='num_lab_procedures')
plt.show()

# Understanding the Target

#### Class Balance
~11% of observations are of the primary target variable of readmitted within 30 days

#### Relationships to A Select Subset of Features
- Readmission rates are slightly higher in patients with diabetesMed, insulin, metformin
- Mixed impact of admission source on readimission
- Numeric features tend to be positively associated with readmission rates (higher value = higher readmission rate)

In [ ]:
# Class Balance
y_train.readmitted.value_counts(normalize=True).round(4)

In [ ]:
# Looking at the readmission rate across features
for f in ['diabetesMed', 'insulin', 'metformin', 'admission_type_id', 'admission_source_id']:
    f_ct = pd.crosstab(X_train[f], y_train['readmitted'])
    print(f_ct.apply(lambda x: x / sum(x), axis=0))

In [ ]:
plotting_df = X_train.loc[:, ['num_lab_procedures', 'num_medications', 'time_in_hospital']]
plotting_df['target'] = y_train['readmitted']

long_plot_df = plotting_df.melt(id_vars='target')

# Creating a grid to see the relationships between the features and the target
target_grid = sns.FacetGrid(long_plot_df, col_wrap=3, col='variable', sharey=False)
target_grid.map_dataframe(sns.boxplot, x='target', y='value')
plt.show()